# Yasi — Environment & Running the CodeCovers **only** Yasi's section of `task.pdf`. Parsa's MMV work lives in a separatenotebook (`parsa_mmv_loss.ipynb`) and shares nothing with this one except the repo.### The checklist this notebook ticks**Get one full pipeline run working**- [ ] `prepare_data` on METABRIC · [ ] `train_tokenizer` · [ ] `train_labeltransform` · [ ] `finetune` → `metrics.json`**Baseline numbers across model types (METABRIC)**- [ ] `metabric/survival` · [ ] `metabric/deephit` · [ ] `metabric/dsm` · [ ] `metabric/mensa`- [ ] all four `metrics.json` collected in one shared folder**Second dataset** — see §4; this one deviates from `task.pdf` on purpose.**Config fluency**- [ ] CLI hyperparameter override · [ ] loss recipe swap via `tasks/losses=`- [ ] read `docs/loss.md` + `docs/mensa.md` · [ ] every command documentedA live tracker at the end prints which boxes actually got ticked.### Start here1. Upload `dmmst/` as a Kaggle **Dataset**, or set `GIT_URL` below.2. Settings → **Internet: On**. GPU optional — METABRIC is 1523 training rows.3. **Leave `SMOKE_TEST = True` for the first pass.** It runs everything at 3 epochs   in about a minute, so you find out that all four models work *before* committing   to the real 500-epoch runs. Flip it off once it's green.

## Options

In [ ]:
# ------------------------------- what to run -------------------------------
BASELINES = ["survival", "deephit", "dsm", "mensa"]   # task.pdf's four models
DATASET   = "metabric"

RUN_BASELINES      = True
RUN_CONFIG_FLUENCY = True
# Extra datasets, run with the same four baselines - see section 4.
#   hsa_synthetic : 5000 rows, 2 EVENTS (multi-event path, per-event metrics)
#   support       : 8873 rows, 14 features, 1 event (named in the paper's Sec. 4.1)
EXTRA_DATASETS = ["hsa_synthetic", "support"]

EXTRA_BASELINES   = ["survival", "deephit", "dsm", "mensa"]
HSA_WITHIN_CINDEX = True      # also run experiments=hsa_synthetic/survival-within-cindex

# ------------------------------- run control -------------------------------
SMOKE_TEST   = True     # True  -> 3 epochs, ~1 min total, proves everything works
SMOKE_EPOCHS = 3
NUM_EPOCHS   = None     # used only when SMOKE_TEST = False; None = config default (500)

SEEDS = [0]
# SEEDS = [0, 1, 2, 3, 4]   # <- uncomment for error bars; required for the paper.
#                              A single seed cannot produce a spread: the sd inside
#                              metrics.json is computed within one run and is always
#                              exactly 0.0. Real spread comes from re-running seeds.

EXTRA_OVERRIDES = []

INSTALL_DEPS = True
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # cloned automatically

RESULTS_DIR = "/kaggle/working/results"

## 1 · Setup

In [ ]:
import os, sys, shutil, subprocess, json, time
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = None

def looks_like_repo(p: Path) -> bool:
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

# 1. already here from an earlier cell run in this session
if looks_like_repo(WORK / "dmmst"):
    REPO = WORK / "dmmst"
    print("Using existing clone.")

# 2. clone it - this is the normal path, no Kaggle Dataset needed
elif GIT_URL:
    dst = WORK / "dmmst"
    print(f"Cloning {GIT_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(dst)], check=True)
    REPO = dst

# 3. fallback: someone attached the folder as a Kaggle Dataset instead
if REPO is None:
    for base in [Path("/kaggle/input"), Path.cwd()]:
        if not base.exists():
            continue
        for cand in sorted(base.rglob("conf")):
            if looks_like_repo(cand.parent):
                dst = WORK / "dmmst"
                if not dst.exists():
                    # /kaggle/input is read-only and the pipeline writes into data/
                    print(f"Copying {cand.parent} -> {dst}")
                    shutil.copytree(cand.parent, dst)
                REPO = dst
                break
        if REPO:
            break

assert REPO is not None, (
    "Could not obtain the code. Set GIT_URL (default should just work), or attach "
    "the dmmst folder as a Kaggle Dataset."
)
REPO = REPO.resolve()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repo:", REPO)

# the datasets ship inside the repo, so nothing else to download
for d, f in [("metabric", "metabric_IHC4_clinical_train_test.h5"),
             ("hsa-synthetic", "simulated_data.csv"),
             ("support", "support_train_test.h5")]:
    p = REPO / "data" / d / f
    print(f"  data/{d:14} {'OK' if p.is_file() else 'MISSING'}")

### DependenciesOnly what the **training path** needs. Three things learned the hard way:- `transformers==4.50.0` is a hard pin — the repo subclasses HF internals in  `sat/models/bert/modeling_bert.py`; 4.57 breaks it.- **`torch` and `numpy` are left alone**, so no forced session restart. `numba` is  unpinned so pip matches Kaggle's numpy.- `nvidia-ml-py`, **not** `nvidia-ml-py3` — the latter is a 2017 wheel that cannot  locate the NVML library and used to crash every GPU run from a *logging* call.`lifelines` is not installed: it is only used by `sat.eda`, never by training, so the`scipy<1.14` pin it drags in is avoided entirely.

In [ ]:
def sh(cmd, check=True):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

if INSTALL_DEPS:
    import torch
    tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    torchsurv_pin = "torchsurv" if tv >= (2, 8) else "torchsurv==0.1.4"
    print(f"torch {torch.__version__} -> {torchsurv_pin}")
    pkgs = [
        "transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22",
        "evaluate>=0.4.3", "accelerate>=1.4.0",
        "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
        torchsurv_pin,
        "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
        "numba", "nvidia-ml-py", "polars", "tqdm",
    ]
    sh("pip install -q " + " ".join(f'"{p}"' for p in pkgs))
    print("\nIf pip changed an already-imported package, restart the session "
          "(Run -> Restart) and re-run with INSTALL_DEPS = False.")
else:
    print("Skipping install.")

### Sanity check

In [ ]:
import importlib, torch
for m in ["transformers", "datasets", "hydra", "torchsurv"]:
    print(f"  {m:<14} {getattr(importlib.import_module(m), '__version__', 'n/a')}")
print(f"  torch          {torch.__version__}  cuda={torch.cuda.is_available()}")

from sat.loss import SATNLLPCHazardLoss, DeepHitLikelihoodLoss, DSMLoss, MENSALoss
print("\n  all four baseline losses import OK")

os.makedirs(RESULTS_DIR, exist_ok=True)
DONE = {}     # checklist tracker

In [ ]:
import re
from tqdm.auto import tqdm

ENV = os.environ.copy()
ENV["TOKENIZERS_PARALLELISM"] = "false"
ENV["HYDRA_FULL_ERROR"] = "1"
ENV["PYTHONUNBUFFERED"] = "1"      # without this the child buffers and the bar stalls

# lines worth surfacing live; everything else is kept for the failure tail only
_INTERESTING = re.compile(
    r"error|exception|traceback|warning|failed|"
    r"eval_ipcw_weighted_avg|eval_brier_weighted_avg|Save model|Write prediction",
    re.I,
)
_PROBLEM = re.compile(r"error|exception|traceback|failed|warning", re.I)
_EPOCH = re.compile(r"'epoch': ([0-9.]+)")


def expected_epochs():
    """Total epochs for the progress bar, or None if the config decides."""
    if SMOKE_TEST:
        return SMOKE_EPOCHS
    return NUM_EPOCHS      # None -> indeterminate bar (still shows count + elapsed)


def run_sat(script, experiment, overrides=(), label=None, tail_lines=15):
    """Run a sat entry point, streaming progress live.

    subprocess.run(stdout=PIPE) blocks until the process exits, which on a long
    run means staring at an empty cell for 20 minutes. This streams instead, and
    drives a tqdm bar off the trainer's own "'epoch': N" log lines.
    """
    cmd = [sys.executable, "-m", f"sat.{script}", f"experiments={experiment}", *overrides]
    label = label or f"{script} {experiment}"
    print()
    print("=" * 72)
    print(label)
    print("$ " + " ".join(cmd[2:]))
    print("=" * 72)

    total = expected_epochs() if script == "finetune" else None
    bar = tqdm(total=total, desc=label[:40], unit="ep", leave=False,
               bar_format="{l_bar}{bar}| {n:.0f}/{total_fmt} [{elapsed}<{remaining}]"
                          if total else "{desc}: {n:.0f} ep [{elapsed}]")

    t0 = time.time()
    lines = []
    last_shown = [0.0]     # throttle for routine progress lines
    proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, text=True, bufsize=1,
                            errors="replace",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        for raw in proc.stdout:
            # tqdm rewrites one line in place; keep only the final segment
            line = raw.rstrip(chr(10)).split(chr(13))[-1].rstrip()
            if not line:
                continue
            lines.append(line)

            m = _EPOCH.search(line)
            if m:
                ep = float(m.group(1))
                bar.n = min(ep, total) if total else ep
                bar.refresh()

            # Always surface problems. Throttle everything else: with
            # eval_steps=1 a 500-epoch run emits ~1500 metric lines, which would
            # bury the notebook.
            is_problem = _PROBLEM.search(line)
            now = time.time()
            if is_problem or (_INTERESTING.search(line) and now - last_shown[0] > 10):
                bar.write("   " + line[:160])
                if not is_problem:
                    last_shown[0] = now
    finally:
        proc.wait()
        bar.close()

    dt = time.time() - t0
    ok = proc.returncode == 0
    if not ok:
        print("--- last lines ---")
        print(chr(10).join(lines[-tail_lines:]))

    swallowed = sum(1 for l in lines
                    if "Error in survival_predictions" in l or "Invalid predictions" in l)
    if swallowed:
        print(f"  !! {swallowed} swallowed prediction error(s) - metrics may be "
              f"hard-coded fallbacks rather than computed values")
    print(f"--> {'OK' if ok else 'FAILED'} in {dt:.1f}s")
    return ok, dt, chr(10).join(lines)


def train_overrides(seed=0):
    """Common Hydra overrides.

    num_train_epochs lives at trainer.training_arguments.num_train_epochs - a bare
    `num_train_epochs=` override is rejected by Hydra. warmup_steps also gates
    eval_delay, so short runs need it at 0 or no evaluation happens and
    load_best_model_at_end fails.
    """
    ov = [f"seed={seed}"]
    if SMOKE_TEST:
        ov += [f"trainer.training_arguments.num_train_epochs={SMOKE_EPOCHS}",
               "warmup_steps=0"]
    elif NUM_EPOCHS:
        ov += [f"trainer.training_arguments.num_train_epochs={NUM_EPOCHS}"]
    return ov + list(EXTRA_OVERRIDES)

In [ ]:
import pandas as pd

def collect_metrics(dataset, modelname, tag):
    """Copy a run's metrics.json into the shared results folder and parse it.

    metrics.json is NESTED:
        {"validation": {...}, "test": {"ipcw_weighted_avg": {"mean":..,"sd":..}, ...}}
    so a flat parse silently yields nothing.
    """
    src = REPO / "data" / "model-hub" / dataset / modelname / "metrics.json"
    if not src.is_file():
        print(f"  !! no metrics.json at {src}")
        return None
    dst = Path(RESULTS_DIR) / f"{tag}.json"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  saved -> {dst}")
    with open(dst) as f:
        return json.load(f)


def flatten(metrics, split="test"):
    """Pull {metric: mean} out of one split of the nested metrics.json."""
    out = {}
    for k, v in (metrics.get(split) or {}).items():
        if isinstance(v, dict) and "mean" in v:
            out[k] = v["mean"]
        elif isinstance(v, (int, float)):
            out[k] = v
    return out


HEADLINE = ["ipcw_weighted_avg", "brier_weighted_avg", "loss",
            "within_subject_ipcw", "mismatch"]
NOISE = ("runtime", "samples_per_second", "steps_per_second", "n")

def results_table(results, title, split="test"):
    rows = []
    for tag, m in results.items():
        if m is None or m.get("_failed"):
            rows.append({"run": tag, "status": "FAILED"})
            continue
        flat = flatten(m, split)
        row = {"run": tag, "status": "ok"}
        for k in HEADLINE:
            if k in flat:
                row[k] = round(flat[k], 4)
        for k, v in flat.items():          # any remaining per-event metrics
            if k not in row and not k.endswith("_n") and k not in NOISE:
                row[k] = round(v, 4)
        rows.append(row)
    if not rows:
        print("no results yet")
        return None
    df = pd.DataFrame(rows).set_index("run").dropna(axis=1, how="all")
    print(f"\n### {title}  (split = {split})")
    display(df)
    return df


import re as _re

def aggregate_seeds(results, split="test"):
    """Mean +/- sd ACROSS seeds.

    The `variance`/`sd` fields inside a single metrics.json are computed within
    one run, so with one seed they are always exactly 0.0 - that is an artifact,
    not a real spread. Any spread worth reporting comes from re-running with
    different seeds, which is what this aggregates.
    """
    groups = {}
    for tag, m in results.items():
        if m is None or m.get("_failed"):
            continue
        base = _re.sub(r"_seed\d+$", "", tag)
        groups.setdefault(base, []).append(flatten(m, split))
    if not groups:
        print("nothing to aggregate")
        return None

    rows = []
    for base, runs in sorted(groups.items()):
        row = {"run": base, "seeds": len(runs)}
        keys = [k for k in runs[0]
                if not k.endswith("_n") and k not in NOISE]
        for k in HEADLINE + [k for k in keys if k not in HEADLINE]:
            vals = [r[k] for r in runs if k in r]
            if not vals:
                continue
            mean = sum(vals) / len(vals)
            if len(vals) > 1:
                sd = (sum((v - mean) ** 2 for v in vals) / (len(vals) - 1)) ** 0.5
                row[k] = f"{mean:.4f} +/- {sd:.4f}"
            else:
                row[k] = f"{mean:.4f}"
        rows.append(row)

    df = pd.DataFrame(rows).set_index("run")
    print(f"\n### Aggregated across seeds  (split = {split})")
    display(df)
    if all(r["seeds"] == 1 for r in rows):
        print("\nOnly one seed per configuration, so there is no spread to report.\n"
              "Uncomment the multi-seed line in Options before quoting any of these\n"
              "numbers in the paper.")
    return df

## 2 · One full pipeline run`prepare_data` → `train_tokenizer` → `train_labeltransform` produce artifacts that**every** model reuses, so they run once per dataset. Then one `finetune` to prove thechain ends in a `metrics.json`.

In [ ]:
PIPE = f"{DATASET}/survival"

for step in ["prepare_data", "train_tokenizer", "train_labeltransform"]:
    ok, _, _ = run_sat(step, PIPE)
    DONE[step] = ok
    assert ok, f"{step} failed - nothing downstream can work until this passes"

ok, dt, _ = run_sat("finetune", PIPE, train_overrides(SEEDS[0]) + ["modelname=pipeline_check"],
                    label="finetune (pipeline check)")
DONE["finetune"] = ok

m = collect_metrics(DATASET, "pipeline_check", f"{DATASET}_pipeline_check") if ok else None
DONE["metrics.json produced"] = m is not None
if m:
    print("\nmetrics.json splits:", list(m.keys()))
    print("test metrics:", list(flatten(m, "test"))[:6], "...")

## 3 · Baselines across model typesOne `finetune` per model, all four `metrics.json` copied into `RESULTS_DIR`.**`metabric/mensa` did not exist upstream** — only `metabric_numeric/mensa.yaml`, andthat file was malformed (missing `@package _global_`, wrong interpolation namespace, soit could not compose). Both were rewritten for this repo.

In [ ]:
RESULTS = {}

if RUN_BASELINES:
    jobs = [(m, s_) for m in BASELINES for s_ in SEEDS]
    outer = tqdm(jobs, desc="METABRIC baselines", unit="run")
    for model, seed in outer:
        outer.set_postfix_str(f"{model} seed={seed}")
        if True:
            tag = f"{DATASET}_{model}_seed{seed}"
            ok, dt, _ = run_sat("finetune", f"{DATASET}/{model}",
                                train_overrides(seed), label=f"BASELINE · {model} (seed {seed})")
            RESULTS[tag] = collect_metrics(DATASET, model, tag) if ok else {"_failed": True}
            DONE[f"baseline {model}"] = ok
    n_ok = sum(1 for v in RESULTS.values() if v and not v.get("_failed"))
    DONE["all metrics in one folder"] = n_ok == len(BASELINES) * len(SEEDS)
    print(f"\n{n_ok}/{len(RESULTS)} runs produced metrics in {RESULTS_DIR}")
else:
    print("skipped")

In [ ]:
table = results_table(RESULTS, "Baseline comparison — METABRIC") if RUN_BASELINES else None
if SMOKE_TEST:
    print("\nSMOKE_TEST is on: these numbers are from 3 epochs and are NOT meaningful.\n"
          "They only prove the four models run end-to-end. Set SMOKE_TEST = False for\n"
          "numbers that can go in the paper.")

## 4 · The other datasetsThe same four baselines on every dataset in `EXTRA_DATASETS`. All of them ship insidethe repo, so nothing needs downloading.**`hsa_synthetic`** — 5000 rows, **`num_events: 2`**. This is the multi-event path:metrics come back per event (`ipcw_0th_event`, `ipcw_1th_event`, …) and the**within-subject C-index** becomes meaningful — the metric tied to the paper's `L_mul`.> Read the numbers, not just the checkmarks. This dataset is inherited from the> abandoned earlier work; its event offsets are shared across individuals, so every> subject has essentially the same event ordering, and a model emitting one constant> ordering scores well without learning anything. Evidence that the *pipeline* works,> not that the *method* does.**`support`** — 8873 patients, 14 features, single event. One of the three tabulardatasets named in the paper's §4.1 (METABRIC / SUPPORT / SEER) and the standardbenchmark in the SurvTrace comparison table, so these numbers are directly comparableto published results.*SEER, the third, needs a signed data-request agreement and cannot be bundled.*

In [ ]:
HSA = {}   # results for every extra dataset

for DS in EXTRA_DATASETS:
    bar = "#" * 72
    print()
    print(bar)
    print(f"#  DATASET: {DS}")
    print(bar)
    ok_all = True
    for step in ["prepare_data", "train_tokenizer", "train_labeltransform"]:
        ok, _, _ = run_sat(step, f"{DS}/survival")
        ok_all &= ok
    DONE[f"{DS} pipeline"] = ok_all
    if not ok_all:
        print(f"  !! preprocessing failed for {DS}, skipping its models")
        continue

    jobs = [(m, s_) for m in EXTRA_BASELINES for s_ in SEEDS]
    outer = tqdm(jobs, desc=f"{DS} baselines", unit="run")
    for model, seed in outer:
        outer.set_postfix_str(f"{model} seed={seed}")
        if True:
            tag = f"{DS}_{model}_seed{seed}"
            ok, _, _ = run_sat("finetune", f"{DS}/{model}", train_overrides(seed),
                               label=f"{DS} · {model} (seed {seed})")
            HSA[tag] = collect_metrics(DS, model, tag) if ok else {"_failed": True}
            DONE[f"{DS} baseline {model}"] = ok

    # the multi-event metric only exists for the multi-event dataset
    if DS == "hsa_synthetic" and HSA_WITHIN_CINDEX:
        for seed in SEEDS:
            tag = f"{DS}_within_cindex_seed{seed}"
            ok, _, _ = run_sat("finetune", f"{DS}/survival-within-cindex",
                               train_overrides(seed) + ["modelname=survival_within"],
                               label=f"{DS} · within-subject C-index (seed {seed})")
            HSA[tag] = collect_metrics(DS, "survival_within", tag) if ok else {"_failed": True}
            DONE["hsa within-subject c-index"] = ok

In [ ]:
if HSA:
    for DS in EXTRA_DATASETS:
        subset = {k: v for k, v in HSA.items() if k.startswith(DS + "_")}
        if subset:
            results_table(subset, f"{DS}")
    # surface the multi-event metric explicitly
    for tag, m in HSA.items():
        if m and not m.get("_failed"):
            w = {k: v for k, v in flatten(m, "test").items() if "within_subject" in k}
            if w:
                print()
                print(f"within-subject C-index - {tag}")
                for k, v in sorted(w.items()):
                    print(f"  {k:<28} {v:.4f}")

## 5 · Config fluencyTwo things `task.pdf` asks you to demonstrate.**Override `modelname=` whenever you change anything.** Output goes to`data/model-hub/<dataset>/<modelname>/`, so without it every variant overwrites theprevious run's `metrics.json` and you silently compare a run against itself.

In [ ]:
FLUENCY = {}

if RUN_CONFIG_FLUENCY:
    # (a) override a hyperparameter from the CLI
    ok, _, _ = run_sat("finetune", f"{DATASET}/survival",
                       train_overrides(SEEDS[0]) + ["learning_rate=0.001",
                                                    "modelname=survival_lr001"],
                       label="FLUENCY · learning_rate=0.001")
    FLUENCY["lr=0.001"] = collect_metrics(DATASET, "survival_lr001",
                                          f"{DATASET}_survival_lr001") if ok else {"_failed": True}
    DONE["CLI hyperparameter override"] = ok

    # (b) swap the loss recipe - this one is the paper's own L_PCH + L_rank + L_mul
    ok, _, _ = run_sat("finetune", f"{DATASET}/survival",
                       train_overrides(SEEDS[0]) + ["tasks/losses=nllpch_sample_event_ranking",
                                                    "modelname=survival_ranking"],
                       label="FLUENCY · tasks/losses=nllpch_sample_event_ranking")
    FLUENCY["L_PCH+L_rank+L_mul"] = collect_metrics(DATASET, "survival_ranking",
                                                    f"{DATASET}_survival_ranking") if ok else {"_failed": True}
    DONE["loss recipe swap"] = ok

    results_table(FLUENCY, "Config-fluency runs")
else:
    print("skipped")

### Required reading`docs/loss.md` and `docs/mensa.md` — rendered below so the checkbox is genuinely tickedrather than just asserted.

In [ ]:
from IPython.display import Markdown, display as _d
for doc in ["docs/loss.md", "docs/mensa.md"]:
    p = REPO / doc
    if p.is_file():
        print(f"\n{'#' * 72}\n# {doc}\n{'#' * 72}")
        _d(Markdown(p.read_text(encoding="utf8")))
        DONE[f"read {doc}"] = True
    else:
        print(f"MISSING: {doc}")
        DONE[f"read {doc}"] = False

## 6 · Experimental setup`task.pdf`: *"Documented the exact commands/configs used for every run above (this becomesthe paper's experimental setup section)."*This writes `experimental_setup.md` into the results folder — every command, the resolvedconfig for each run, and the environment. Download it from the Kaggle output panel.

In [ ]:
ALL = {}
ALL.update(RESULTS)
ALL.update({f"fluency::{k}": v for k, v in FLUENCY.items()})
ALL.update(HSA)

out = Path(RESULTS_DIR)
lines = ["# Experimental setup", "",
         "Auto-generated by `yasi_environment_and_baselines.ipynb`.", ""]

import platform, torch, transformers
lines += ["## Environment", "",
          f"- python {platform.python_version()}, {platform.system()}",
          f"- torch {torch.__version__} (cuda={torch.cuda.is_available()})",
          f"- transformers {transformers.__version__}",
          f"- mode: {'SMOKE TEST - %d epochs, numbers NOT publishable' % SMOKE_EPOCHS if SMOKE_TEST else 'full run'}",
          f"- seeds: {SEEDS}", ""]

lines += ["## Commands", "", "### Shared preprocessing (once per dataset)", "```bash"]
for step in ["prepare_data", "train_tokenizer", "train_labeltransform"]:
    lines.append(f"python -m sat.{step} experiments={DATASET}/survival")
lines += ["```", "", "### Baselines", "```bash"]
for model in BASELINES:
    for seed in SEEDS:
        lines.append(f"python -m sat.finetune experiments={DATASET}/{model} seed={seed}")
lines += ["```", "", "### Config fluency", "```bash",
          f"python -m sat.finetune experiments={DATASET}/survival learning_rate=0.001 modelname=survival_lr001",
          f"python -m sat.finetune experiments={DATASET}/survival tasks/losses=nllpch_sample_event_ranking modelname=survival_ranking",
          "```", ""]
for DS in EXTRA_DATASETS:
    lines += [f"### {DS}", "```bash",
              f"python -m sat.prepare_data experiments={DS}/survival",
              f"python -m sat.train_tokenizer experiments={DS}/survival",
              f"python -m sat.train_labeltransform experiments={DS}/survival"]
    for model in EXTRA_BASELINES:
        for seed in SEEDS:
            lines.append(f"python -m sat.finetune experiments={DS}/{model} seed={seed}")
    if DS == "hsa_synthetic" and HSA_WITHIN_CINDEX:
        lines.append(f"python -m sat.finetune experiments={DS}/survival-within-cindex "
                     "modelname=survival_within")
    lines += ["```", ""]

lines += ["## Results (test split)", ""]
df = results_table(ALL, "All runs")
if df is not None:
    df.to_csv(out / "all_results.csv")
    lines += [df.to_markdown(), ""]

agg = aggregate_seeds(ALL)
if agg is not None:
    agg.to_csv(out / "aggregated_by_seed.csv")
    lines += ["### Aggregated across seeds", "", agg.to_markdown(), ""]
    if len(SEEDS) == 1:
        lines += ["> Single seed - no spread. Uncomment the multi-seed line in the",
                  "> notebook's Options before quoting these numbers.", ""]

lines += ["## Caveats", "",
          "- **HSA-synthetic numbers are a pipeline check, not evidence.** The dataset is",
          "  inherited from abandoned work and its event offsets are shared across",
          "  individuals, so every subject has essentially the same event ordering. A model",
          "  emitting one constant ordering scores well on within-subject concordance",
          "  without learning anything. A purpose-built simulator is still outstanding.",
          "- **`metabric/mensa` did not exist upstream** and was written for this repo;",
          "  `metabric_numeric/mensa.yaml` existed but was malformed and was rewritten.",
          f"- Seeds run: {SEEDS}." + ("  Single seed - no spread is reportable."
                                      if len(SEEDS) == 1 else ""), ""]

(out / "experimental_setup.md").write_text("\n".join(lines), encoding="utf8")
with open(out / "summary.json", "w") as f:
    json.dump(ALL, f, indent=2)
print(f"\nWrote {out/'experimental_setup.md'}")
for p in sorted(out.glob("*")):
    print("  ", p.name)

## Checklist

In [ ]:
print("task.pdf - Yasi\n" + "=" * 46)
groups = {
    "One full pipeline run": ["prepare_data", "train_tokenizer",
                              "train_labeltransform", "finetune",
                              "metrics.json produced"],
    "Baselines (METABRIC)":  [f"baseline {m}" for m in BASELINES]
                             + ["all metrics in one folder"],
    "Other datasets":        [k for DS in EXTRA_DATASETS
                                for k in ([f"{DS} pipeline"]
                                          + [f"{DS} baseline {m}" for m in EXTRA_BASELINES])]
                             + ["hsa within-subject c-index"],
    "Config fluency":        ["CLI hyperparameter override", "loss recipe swap",
                              "read docs/loss.md", "read docs/mensa.md"],
}
for g, keys in groups.items():
    print(f"\n{g}")
    for k in keys:
        v = DONE.get(k)
        print(f"  [{'x' if v else ' '}] {k}" + ("" if v is not None else "   (not run)"))

print("\n" + "=" * 46)
print(f"documented commands -> {RESULTS_DIR}/experimental_setup.md")
if SMOKE_TEST:
    print("\nSTILL A SMOKE TEST. Everything above ran, but at 3 epochs.")
    print("Set SMOKE_TEST = False and re-run for numbers that can go in the paper.")

## Notes**Runtime.** METABRIC is small, but the config is 500 epochs with evaluation *and*checkpointing every single step — expect roughly 10–20 minutes per model for a real run,mostly eval/save overhead rather than compute. Four models plus fluency runs is a coupleof hours. `conf/trainer/metabric/speedy.yaml` evaluates per epoch instead.**Error bars.** A single seed gives `variance: 0.0` in `metrics.json`, which is anartifact of one run, not a real result. Use `SEEDS = [0,1,2,3,4]`; `sat.ci` (bootstrap)and `sat.cv` (k-fold) are also available.**Two upstream bugs were fixed to make this work** — both would have hit you here:- `log_gpu_utilization()` called `nvmlInit()` unguarded, so a *logging* call aborted  every GPU run when NVML was unavailable. Now fail-safe, and `requirements` moved from  the abandoned `nvidia-ml-py3` to `nvidia-ml-py`.- The DSM head padded `hazard` one column wider than `risk`/`survival`. The shape check  in `survival_predictions` then threw, the exception was swallowed, and Brier/IPCW  silently returned their hard-coded `0.5` defaults. DSM appeared to work while  reporting fabricated numbers straight into the comparison table.